# SPY 日内动量策略（vectorbt 版）

本 notebook 实现 `intraday_momentum_strategy_reproduction_spec.md`
中描述的 SPY 日内动量策略复现。

这个版本包含：

- 使用 WSL 里的 `rq` 环境内核（`rqwsl`），该环境已经安装 `vectorbt`、`pyarrow` 和 `rqdatac`。
- 从 Polygon 获取标准化后的 SPY 或其他美股 ETF OHLCV 数据，或者读取本地 CSV。
- 将获取/标准化后的数据和回测输出保存为 parquet 文件，和本项目现有缓存风格保持一致。
- 计算“同一日内时刻”的历史噪声带，并避免使用当前交易日信息。
- 支持 SPY baseline 的 `boundary_and_vwap` 出场，也支持改进论文里的 VWAP、不同出场边界、边界+VWAP、阶梯止盈止损、VWAP+阶梯、边界+阶梯等出场族。
- 先生成事件驱动的订单流水，再用 vectorbt 检查订单/交易记录，同时保留一条支持杠杆目标的内部权益曲线。

运行时请使用 `rqwsl` 内核。如果 Jupyter 当前显示的内核不是
`rqwsl`，先切换内核，再运行数据和回测单元。


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Literal
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

try:
    import vectorbt as vbt
except ImportError as exc:
    raise ImportError(
        "需要安装 vectorbt。本 WSL 项目请使用 Jupyter 内核 "
        "'rqwsl'，对应 /home/pl_ubuntu/miniforge3/envs/rq/bin/python。"
    ) from exc


NY_TZ = "America/New_York"
RQ_ENV_PYTHON = "/home/pl_ubuntu/miniforge3/envs/rq/bin/python"

print("Python:", sys.executable)
print("vectorbt:", vbt.__version__)
if sys.executable != RQ_ENV_PYTHON:
    print(
        "提示：本 notebook 已按 rq 环境检查。"
        f"预期 Python 路径：{RQ_ENV_PYTHON}."
    )


## 1. 参数配置

默认配置对应 spec 里的原始 SPY baseline：

- SPY 1 分钟 K 线。
- 14 个交易日的“同一日内时刻”噪声回看窗口。
- 入场信号使用 VWAP 过滤。
- 目标日波动率 2%，最大总敞口限制为 4 倍。
- 从纽约时间 10:00 开始，每 30 分钟检查一次入场。
- 收盘前最后 5 分钟之前强制平仓。
- 使用 `boundary_and_vwap` 出场。

`missing_bar_policy="drop_day"` 会删除存在日内缺失 bar 的交易日。
这是更严格的复现选择。如果只是探索，希望保留供应商数据中不完美的交易日，
可以改成 `"keep_approx"`；notebook 会把这些行标记为
`session_approximate=True`。


In [ ]:
from __future__ import annotations

ExitVariant = Literal[
    "opposite_or_eod",
    "vwap",
    "boundary_different_exit",
    "boundary_different_exit_and_vwap",
    "boundary_and_vwap",
    "ladder",
    "vwap_and_ladder",
    "boundary_and_ladder",
]


@dataclass(frozen=True)
class StrategyConfig:
    symbol: str = "SPY"
    start_date: str = "2024-01-02"
    end_date: str = "2024-03-28"
    bar_size: str = "1min"
    data_source: Literal["polygon", "csv"] = "polygon"
    cache_dir: str = "data/spy"
    results_dir: str = "results/spy_intraday_momentum"
    intraday_csv: str | None = None
    daily_csv: str | None = None
    dividends_csv: str | None = None
    adjusted: bool = True
    regular_hours_only: bool = True
    cache_csv_inputs: bool = True
    missing_bar_policy: Literal["drop_day", "keep_approx"] = "drop_day"

    lookback_days: int = 14
    volatility_multiplier_entry: float = 1.0
    volatility_multiplier_exit: float | None = None
    entry_vwap_filter: bool = True
    target_daily_volatility: float = 0.02
    daily_vol_window: int = 15
    max_leverage: float = 4.0

    start_trade_after_open_minutes: int = 30
    trade_frequency_minutes: int = 30
    exit_trades_before_close_minutes: int = 5
    exit_variant: ExitVariant = "boundary_and_vwap"
    allow_reversal: bool = True
    execution_mode: Literal["signal_bar_close", "next_bar_open"] = "signal_bar_close"

    # 阶梯参数是相对于入场价的绝对价格差。
    # 只有当 exit_variant 包含 "ladder" 时才需要填写。
    stop_loss_ladder_step_0_diff: float | None = None
    stop_loss_ladder_step_1_diff: float | None = None
    take_profit_ladder_step_0_diff: float | None = None
    take_profit_ladder_step_1_diff: float | None = None
    take_profit_fraction_step_0: float = 0.50

    initial_equity: float = 100_000.0
    transaction_cost_bps_one_way: float = 0.0
    slippage_bps_one_way: float = 0.0


CONFIG = StrategyConfig()

REALISTIC_CONFIG = replace(
    CONFIG,
    execution_mode="next_bar_open",
    transaction_cost_bps_one_way=0.5,
    slippage_bps_one_way=0.5,
)

FULL_REPRODUCTION_CONFIG = replace(
    CONFIG,
    start_date="2014-10-01",
    end_date="2024-10-01",
)

# 阶梯配置示例。这里沿用 QQQ 论文风格的美元距离，
# 对 SPY 来说只适合作为机制检查，正式使用前需要重新优化。
LADDER_EXAMPLE_CONFIG = replace(
    CONFIG,
    exit_variant="vwap_and_ladder",
    stop_loss_ladder_step_0_diff=-0.41,
    stop_loss_ladder_step_1_diff=-0.28,
    take_profit_ladder_step_0_diff=2.20,
    take_profit_ladder_step_1_diff=37.08,
)

CONFIG


## 2. 数据请求与 Parquet 缓存

策略引擎只消费标准化 DataFrame。Polygon 在这里仅仅是一个数据适配器。
缓存统一使用 parquet，和本项目其他 `data_rq`、`results` 文件保持一致。

运行 Polygon 拉数前，需要先设置环境变量 `POLYGON_API_KEY`。
如果离线运行，可以设置 `data_source="csv"`，并提供 `intraday_csv`、
`daily_csv`，以及可选的 `dividends_csv`；如果 `cache_csv_inputs=True`，
标准化后的 CSV 输入也会另存为 parquet。


In [ ]:
from __future__ import annotations

def date_token(date_text: str) -> str:
    return date_text.replace("-", "")


def bar_size_to_polygon(bar_size: str) -> tuple[int, str, pd.Timedelta]:
    text = bar_size.lower().strip()
    if text.endswith("min"):
        value = int(text[:-3] or "1")
        return value, "minute", pd.Timedelta(minutes=value)
    if text.endswith("s"):
        value = int(text[:-1] or "1")
        return value, "second", pd.Timedelta(seconds=value)
    if text.endswith("d"):
        value = int(text[:-1] or "1")
        return value, "day", pd.Timedelta(days=value)
    raise ValueError("只支持类似 '1min'、'30min'、'1s'、'1d' 的 bar_size。")


def parse_ny_timestamp(values: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(values, errors="coerce")
    try:
        if parsed.dt.tz is None:
            return parsed.dt.tz_localize(NY_TZ)
        return parsed.dt.tz_convert(NY_TZ)
    except AttributeError:
        parsed = pd.to_datetime(values, errors="coerce", utc=True)
        return parsed.dt.tz_convert(NY_TZ)


def normalize_date_series(values: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(values, errors="coerce")
    try:
        if parsed.dt.tz is not None:
            parsed = parsed.dt.tz_convert(NY_TZ).dt.tz_localize(None)
    except AttributeError:
        parsed = pd.to_datetime(values, errors="coerce", utc=True)
        parsed = parsed.dt.tz_convert(NY_TZ).dt.tz_localize(None)
    return parsed.dt.normalize()


def cache_paths(config: StrategyConfig) -> dict[str, Path]:
    cache_dir = Path(config.cache_dir)
    start = date_token(config.start_date)
    end = date_token(config.end_date)
    bar = config.bar_size.replace(" ", "")
    prefix = f"{config.symbol}_{bar}_{start}_{end}"
    return {
        "intraday": cache_dir / f"{prefix}_intraday.parquet",
        "daily": cache_dir / f"{config.symbol}_{start}_{end}_daily.parquet",
        "dividends": cache_dir / f"{config.symbol}_{start}_{end}_dividends.parquet",
        "meta": cache_dir / f"{prefix}_cache_meta.json",
    }


def read_cached_frame(path: Path, date_cols: list[str]) -> pd.DataFrame | None:
    if not path.exists():
        return None
    frame = pd.read_parquet(path)
    for col in date_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


def write_parquet_frame(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        frame.to_parquet(path, index=False)
    except ImportError as exc:
        raise RuntimeError("写入 parquet 需要安装 pyarrow 或 fastparquet。") from exc
    except ValueError as exc:
        message = str(exc).lower()
        if "parquet" in message or "pyarrow" in message or "fastparquet" in message:
            raise RuntimeError("写入 parquet 需要安装 pyarrow 或 fastparquet。") from exc
        raise


def write_cache_meta(config: StrategyConfig, paths: dict[str, Path]) -> None:
    paths["meta"].parent.mkdir(parents=True, exist_ok=True)
    meta = {
        "config": asdict(config),
        "storage": "parquet",
        "files": {key: str(value) for key, value in paths.items()},
        "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    }
    paths["meta"].write_text(json.dumps(meta, indent=2), encoding="utf-8")


def polygon_get_json(url: str, params: dict | None = None, api_key: str | None = None) -> dict:
    api_key = api_key or os.getenv("POLYGON_API_KEY")
    if not api_key:
        raise RuntimeError("拉取 Polygon 数据前请先设置 POLYGON_API_KEY。")

    params = dict(params or {})
    if "apiKey" not in params:
        params["apiKey"] = api_key
    sep = "&" if "?" in url else "?"
    full_url = f"{url}{sep}{urlencode(params)}"
    request = Request(full_url, headers={"User-Agent": "spy-intraday-momentum-notebook"})
    with urlopen(request, timeout=90) as response:
        return json.loads(response.read().decode("utf-8"))


def fetch_polygon_aggs(
    symbol: str,
    start_date: str,
    end_date: str,
    bar_size: str,
    adjusted: bool = True,
) -> pd.DataFrame:
    multiplier, timespan, _ = bar_size_to_polygon(bar_size)
    url = (
        f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/"
        f"{multiplier}/{timespan}/{start_date}/{end_date}"
    )
    params = {
        "adjusted": str(adjusted).lower(),
        "sort": "asc",
        "limit": 50_000,
    }

    rows: list[dict] = []
    while url:
        payload = polygon_get_json(url, params=params)
        rows.extend(payload.get("results", []))
        url = payload.get("next_url")
        params = {}
        if url:
            time.sleep(0.12)

    if not rows:
        raise RuntimeError(f"Polygon 没有返回聚合行情数据：{symbol}.")

    data = pd.DataFrame(rows).rename(
        columns={
            "t": "timestamp",
            "o": "open",
            "h": "high",
            "l": "low",
            "c": "close",
            "v": "volume",
            "vw": "vwap_vendor",
            "n": "transactions",
        }
    )
    data["timestamp"] = pd.to_datetime(data["timestamp"], unit="ms", utc=True).dt.tz_convert(NY_TZ)
    data["symbol"] = symbol
    keep = [
        "timestamp",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "vwap_vendor",
        "transactions",
    ]
    return data[[col for col in keep if col in data.columns]].sort_values("timestamp")


def fetch_polygon_daily(
    symbol: str,
    start_date: str,
    end_date: str,
    adjusted: bool = True,
) -> pd.DataFrame:
    rows = fetch_polygon_aggs(symbol, start_date, end_date, "1d", adjusted=adjusted)
    daily = rows.rename(columns={"timestamp": "date"}).copy()
    daily["date"] = pd.to_datetime(daily["date"]).dt.tz_convert(NY_TZ).dt.date
    daily["date"] = pd.to_datetime(daily["date"])
    return daily[["date", "symbol", "open", "high", "low", "close", "volume"]].sort_values("date")


def fetch_polygon_dividends(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://api.polygon.io/v3/reference/dividends"
    params = {
        "ticker": symbol,
        "ex_dividend_date.gte": start_date,
        "ex_dividend_date.lte": end_date,
        "limit": 1000,
        "sort": "ex_dividend_date",
        "order": "asc",
    }
    rows: list[dict] = []
    while url:
        payload = polygon_get_json(url, params=params)
        rows.extend(payload.get("results", []))
        url = payload.get("next_url")
        params = {}
        if url:
            time.sleep(0.12)

    if not rows:
        return pd.DataFrame(columns=["date", "cash_amount"])

    dividends = pd.DataFrame(rows)
    dividends["date"] = pd.to_datetime(dividends["ex_dividend_date"])
    amount_col = "cash_amount" if "cash_amount" in dividends.columns else "amount"
    dividends["cash_amount"] = pd.to_numeric(dividends[amount_col], errors="coerce").fillna(0.0)
    return dividends[["date", "cash_amount"]].sort_values("date")


def keep_regular_hours(intraday: pd.DataFrame) -> pd.DataFrame:
    data = intraday.copy()
    data["timestamp"] = parse_ny_timestamp(data["timestamp"])
    local_time = data["timestamp"].dt.time
    start = pd.Timestamp("09:30").time()
    end = pd.Timestamp("16:00").time()
    mask = (local_time >= start) & (local_time < end)
    return data.loc[mask].sort_values("timestamp").reset_index(drop=True)


def fetch_symbol_intraday(
    symbol: str,
    start_date: str,
    end_date: str,
    bar_size: str = "1min",
    adjusted: bool = True,
    regular_hours_only: bool = True,
) -> pd.DataFrame:
    intraday = fetch_polygon_aggs(symbol, start_date, end_date, bar_size, adjusted=adjusted)
    if regular_hours_only:
        intraday = keep_regular_hours(intraday)
    return intraday


def fetch_symbol_daily(symbol: str, start_date: str, end_date: str, adjusted: bool = True) -> pd.DataFrame:
    return fetch_polygon_daily(symbol, start_date, end_date, adjusted=adjusted)


def fetch_spy_intraday(
    start_date: str,
    end_date: str,
    bar_size: str = "1min",
    adjusted: bool = True,
    regular_hours_only: bool = True,
) -> pd.DataFrame:
    return fetch_symbol_intraday("SPY", start_date, end_date, bar_size, adjusted, regular_hours_only)


def fetch_spy_daily(start_date: str, end_date: str, adjusted: bool = True) -> pd.DataFrame:
    return fetch_symbol_daily("SPY", start_date, end_date, adjusted=adjusted)


In [ ]:
from __future__ import annotations

def load_ohlcv_csv(path: str, symbol: str, timestamp_col: str = "timestamp") -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=[timestamp_col])
    frame = frame.rename(columns={timestamp_col: "timestamp"})
    frame["timestamp"] = parse_ny_timestamp(frame["timestamp"])
    if "symbol" not in frame.columns:
        frame["symbol"] = symbol
    return frame.sort_values("timestamp")


def load_daily_csv(path: str, symbol: str) -> pd.DataFrame:
    daily = pd.read_csv(path, parse_dates=["date"])
    if "symbol" not in daily.columns:
        daily["symbol"] = symbol
    return daily.sort_values("date")


def load_dividends_csv(path: str | None) -> pd.DataFrame:
    if not path:
        return pd.DataFrame(columns=["date", "cash_amount"])
    dividends = pd.read_csv(path, parse_dates=["date"])
    if "cash_amount" not in dividends.columns:
        raise ValueError("分红 CSV 必须包含 cash_amount 列。")
    return dividends[["date", "cash_amount"]].sort_values("date")


def load_or_fetch_spy_data(config: StrategyConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths = cache_paths(config)

    if config.data_source == "csv":
        if not config.intraday_csv or not config.daily_csv:
            raise ValueError("CSV 模式必须提供 intraday_csv 和 daily_csv。")
        intraday = load_ohlcv_csv(config.intraday_csv, config.symbol)
        daily = load_daily_csv(config.daily_csv, config.symbol)
        dividends = load_dividends_csv(config.dividends_csv)
        if config.regular_hours_only:
            intraday = keep_regular_hours(intraday)
        if config.cache_csv_inputs:
            write_parquet_frame(intraday, paths["intraday"])
            write_parquet_frame(daily, paths["daily"])
            write_parquet_frame(dividends, paths["dividends"])
            write_cache_meta(config, paths)
        return intraday, daily, dividends

    intraday = read_cached_frame(paths["intraday"], ["timestamp"])
    daily = read_cached_frame(paths["daily"], ["date"])
    dividends = read_cached_frame(paths["dividends"], ["date"])

    if intraday is None:
        intraday = fetch_symbol_intraday(
            config.symbol,
            config.start_date,
            config.end_date,
            bar_size=config.bar_size,
            adjusted=config.adjusted,
            regular_hours_only=config.regular_hours_only,
        )
        write_parquet_frame(intraday, paths["intraday"])

    if daily is None:
        daily = fetch_symbol_daily(config.symbol, config.start_date, config.end_date, adjusted=config.adjusted)
        write_parquet_frame(daily, paths["daily"])

    if dividends is None:
        dividends = fetch_polygon_dividends(config.symbol, config.start_date, config.end_date)
        write_parquet_frame(dividends, paths["dividends"])

    write_cache_meta(config, paths)
    return intraday, daily, dividends


## 3. 数据校验、VWAP 与特征工程

这里实现几个关键的无未来函数规则：

- `sigma_move[d, t]` 只使用当前交易日之前、同一日内时刻的历史绝对波动。
- `sigma_daily[d]` 会整体向后移动一天，所以当天仓位只使用开盘前已知信息。
- VWAP 在每个交易日内累计计算，并在下一个交易日重新开始。
- 入场只在固定时间表上的已完成 bar 检查。

提前收盘日使用该交易日实际可见的最后一个常规交易时段 bar 处理。
对于日内缺失 bar，会根据 `missing_bar_policy` 选择整日删除或标记为近似。


In [ ]:
from __future__ import annotations

def annotate_session_quality(data: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
    _, _, bar_delta = bar_size_to_polygon(config.bar_size)
    pieces = []
    dropped_dates = []

    for date, day in data.groupby("date", sort=True):
        day = day.sort_values("timestamp").copy()
        ts = pd.DatetimeIndex(day["timestamp"])
        expected = pd.date_range(ts.min(), ts.max(), freq=bar_delta)
        missing_count = int(len(expected.difference(ts)))
        day["bars_in_session"] = len(day)
        day["internal_missing_bar_count"] = missing_count
        day["session_approximate"] = missing_count > 0

        if missing_count and config.missing_bar_policy == "drop_day":
            dropped_dates.append(date)
            continue
        pieces.append(day)

    if dropped_dates:
        print(f"已删除 {len(dropped_dates)} 个存在日内缺失 bar 的交易日。")
    if not pieces:
        raise ValueError("数据质量过滤后没有剩余交易日。")
    return pd.concat(pieces, axis=0).sort_values("timestamp").reset_index(drop=True)


def validate_intraday(intraday: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
    required = {"timestamp", "open", "high", "low", "close", "volume"}
    missing = required - set(intraday.columns)
    if missing:
        raise ValueError(f"日内数据缺少字段：{sorted(missing)}")

    data = intraday.copy()
    data["timestamp"] = parse_ny_timestamp(data["timestamp"])
    data = data.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)

    price_cols = ["open", "high", "low", "close"]
    for col in price_cols + ["volume"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    positive_prices = data[price_cols].gt(0).all(axis=1)
    valid_ohlc = (
        data["high"].ge(data[["open", "close"]].max(axis=1))
        & data["low"].le(data[["open", "close"]].min(axis=1))
    )
    bad = ~(positive_prices & valid_ohlc)
    if bad.any():
        print(f"正在删除 {int(bad.sum())} 行无效 OHLC 数据。")
        data = data.loc[~bad].copy()

    if config.regular_hours_only:
        data = keep_regular_hours(data)

    local = data["timestamp"].dt.tz_convert(NY_TZ)
    data["date"] = pd.to_datetime(local.dt.date)
    data["bar_time"] = local.dt.strftime("%H:%M:%S")
    data["bar_index"] = data.groupby("date").cumcount()

    session_midnight = local.dt.normalize()
    session_open_ts = session_midnight + pd.Timedelta(hours=9, minutes=30)
    data["session_open_ts"] = session_open_ts
    data["seconds_from_open"] = (
        (data["timestamp"] - data["session_open_ts"]) / pd.Timedelta(seconds=1)
    ).round().astype(int)
    data["minutes_from_open"] = data["seconds_from_open"] / 60.0

    data = annotate_session_quality(data, config)

    _, _, bar_delta = bar_size_to_polygon(config.bar_size)
    last_bar_ts = data.groupby("date")["timestamp"].transform("max")
    data["session_close_ts"] = last_bar_ts + bar_delta
    data["seconds_to_close"] = (
        (data["session_close_ts"] - data["timestamp"]) / pd.Timedelta(seconds=1)
    ).round().astype(int)
    data["minutes_to_close"] = data["seconds_to_close"] / 60.0

    counts = data.groupby("date").size()
    approx_sessions = data.groupby("date")["session_approximate"].max().sum()
    print(
        f"已加载 {len(data):,} 条日内 bar，覆盖 {len(counts):,} 个交易日。"
        f"每个交易日 bar 数：中位数={counts.median():.0f}，最少={counts.min()}，最多={counts.max()}，"
        f"近似交易日数量={int(approx_sessions)}。"
    )
    return data.reset_index(drop=True)


def add_intraday_vwap(intraday: pd.DataFrame) -> pd.DataFrame:
    data = intraday.copy()
    typical = (data["high"] + data["low"] + data["close"]) / 3.0
    non_negative_volume = data["volume"].clip(lower=0)
    dollar_volume = typical * non_negative_volume
    cum_dollar = dollar_volume.groupby(data["date"]).cumsum()
    cum_volume = non_negative_volume.groupby(data["date"]).cumsum()
    vwap_calc = cum_dollar / cum_volume.replace(0, np.nan)
    data["vwap"] = vwap_calc.groupby(data["date"]).ffill()
    if "vwap_vendor" in data.columns:
        data["vwap"] = data["vwap"].fillna(data["vwap_vendor"])
    data["vwap"] = data["vwap"].fillna(data["close"])
    return data


def prepare_daily(
    daily: pd.DataFrame,
    dividends: pd.DataFrame,
    intraday: pd.DataFrame,
    config: StrategyConfig,
) -> pd.DataFrame:
    required = {"date", "open", "high", "low", "close", "volume"}
    missing = required - set(daily.columns)
    if missing:
        raise ValueError(f"日线数据缺少字段：{sorted(missing)}")

    data = daily.copy()
    data["date"] = normalize_date_series(data["date"])
    data = data.drop_duplicates("date").sort_values("date")

    div = dividends.copy()
    if div.empty:
        div = pd.DataFrame(columns=["date", "cash_amount"])
    div["date"] = normalize_date_series(div["date"])
    div_amount = div.groupby("date")["cash_amount"].sum()

    data["dividend_amount"] = data["date"].map(div_amount).fillna(0.0)
    data["prev_close_raw"] = data["close"].shift(1)
    data["prev_close_for_boundary"] = data["prev_close_raw"] - data["dividend_amount"]
    data["ret_daily"] = data["close"].pct_change()
    data["sigma_daily"] = data["ret_daily"].rolling(
        config.daily_vol_window,
        min_periods=config.daily_vol_window,
    ).std().shift(1)

    intraday_dates = pd.Index(pd.to_datetime(intraday["date"]).unique(), name="date")
    data = data.set_index("date").reindex(intraday_dates).sort_index().reset_index()

    # 如果外部日线文件和日内数据从同一天开始，
    # this fallback keeps the boundary usable after day one. For exact SPY
    # reproduction, include one prior daily bar before start_date.
    session_close = intraday.groupby("date")["close"].last().sort_index()
    fallback_prev_close = session_close.shift(1)
    data["prev_close_for_boundary"] = data["prev_close_for_boundary"].fillna(
        data["date"].map(fallback_prev_close)
    )
    data["dividend_amount"] = data["dividend_amount"].fillna(0.0)
    return data


def add_strategy_features(
    intraday_raw: pd.DataFrame,
    daily_raw: pd.DataFrame,
    dividends: pd.DataFrame,
    config: StrategyConfig,
) -> pd.DataFrame:
    bars = validate_intraday(intraday_raw, config)
    bars = add_intraday_vwap(bars)
    daily = prepare_daily(daily_raw, dividends, bars, config)

    daily_cols = daily[
        ["date", "prev_close_for_boundary", "sigma_daily", "dividend_amount"]
    ].copy()
    bars = bars.merge(daily_cols, on="date", how="left")
    bars["session_open"] = bars.groupby("date")["open"].transform("first")

    bars["abs_move_from_open"] = (bars["close"] / bars["session_open"] - 1.0).abs()
    move_matrix = bars.pivot(index="date", columns="bar_time", values="abs_move_from_open")
    sigma_matrix = move_matrix.rolling(
        config.lookback_days,
        min_periods=config.lookback_days,
    ).mean().shift(1)
    sigma_long = (
        sigma_matrix.stack()
        .rename("sigma_move")
        .reset_index()
        .rename(columns={"level_1": "bar_time"})
    )
    bars = bars.merge(sigma_long, on=["date", "bar_time"], how="left")

    anchor_high = np.maximum(bars["session_open"], bars["prev_close_for_boundary"])
    anchor_low = np.minimum(bars["session_open"], bars["prev_close_for_boundary"])
    vm_exit = (
        config.volatility_multiplier_exit
        if config.volatility_multiplier_exit is not None
        else config.volatility_multiplier_entry
    )
    bars["upper_entry"] = anchor_high * (1 + config.volatility_multiplier_entry * bars["sigma_move"])
    bars["lower_entry"] = anchor_low * (1 - config.volatility_multiplier_entry * bars["sigma_move"])
    bars["upper_exit"] = anchor_high * (1 + vm_exit * bars["sigma_move"])
    bars["lower_exit"] = anchor_low * (1 - vm_exit * bars["sigma_move"])

    _, _, bar_delta = bar_size_to_polygon(config.bar_size)
    execution_delay_seconds = int(bar_delta / pd.Timedelta(seconds=1)) if config.execution_mode == "next_bar_open" else 0
    start_seconds = int(config.start_trade_after_open_minutes * 60)
    frequency_seconds = int(config.trade_frequency_minutes * 60)
    exit_seconds = int(config.exit_trades_before_close_minutes * 60)
    enough_time = bars["seconds_to_close"] > (exit_seconds + execution_delay_seconds)
    scheduled = (
        (bars["seconds_from_open"] >= start_seconds)
        & enough_time
        & ((bars["seconds_from_open"] - start_seconds) % frequency_seconds == 0)
    )
    valid_inputs = (
        bars["sigma_move"].notna()
        & bars["sigma_daily"].gt(0)
        & bars["prev_close_for_boundary"].notna()
    )
    bars["entry_allowed"] = scheduled & valid_inputs

    raw_forced = bars["seconds_to_close"] <= exit_seconds
    bars["forced_exit_bar"] = raw_forced & ~raw_forced.groupby(bars["date"]).shift(fill_value=False)

    bars["gross_exposure_fraction"] = (
        config.target_daily_volatility / bars["sigma_daily"]
    ).clip(upper=config.max_leverage)
    bars.loc[~valid_inputs, "gross_exposure_fraction"] = 0.0

    long_signal = bars["close"] > bars["upper_entry"]
    short_signal = bars["close"] < bars["lower_entry"]
    if config.entry_vwap_filter:
        long_signal &= bars["close"] > bars["vwap"]
        short_signal &= bars["close"] < bars["vwap"]
    bars["long_signal"] = bars["entry_allowed"] & long_signal
    bars["short_signal"] = bars["entry_allowed"] & short_signal

    return bars.set_index("timestamp", drop=False)


## 4. 事件驱动订单

vectorbt 很适合记录订单和交易，但它默认的现金模型不是完整的保证金/杠杆引擎。
这个策略会主动把总敞口目标提高到最高 4 倍，所以 notebook 在事件驱动模拟器内部
维护现金、持仓和权益曲线。主要绩效指标使用这条内部权益曲线。
同一份订单流水仍然会传给 vectorbt，方便查看可读的订单和交易记录。

阶梯出场会在入场后的每一根 bar 检查。如果同一根 bar 同时触及止损和止盈，
按 spec 的保守规则处理：假设先触发止损。


In [ ]:
from __future__ import annotations

@dataclass(frozen=True)
class ExitEvent:
    reason: str
    target_position: int
    raw_price: float
    ladder_step_after: int | None = None
    ambiguous: bool = False


def cost_rate(config: StrategyConfig) -> float:
    return config.transaction_cost_bps_one_way / 10_000.0


def slippage_rate(config: StrategyConfig) -> float:
    return config.slippage_bps_one_way / 10_000.0


def sign(value: int) -> int:
    return 1 if value > 0 else -1 if value < 0 else 0


def is_ladder_variant(config: StrategyConfig) -> bool:
    return config.exit_variant in {"ladder", "vwap_and_ladder", "boundary_and_ladder"}


def require_ladder_params(config: StrategyConfig) -> None:
    if not is_ladder_variant(config):
        return
    required = [
        "stop_loss_ladder_step_0_diff",
        "stop_loss_ladder_step_1_diff",
        "take_profit_ladder_step_0_diff",
        "take_profit_ladder_step_1_diff",
    ]
    missing = [name for name in required if getattr(config, name) is None]
    if missing:
        raise ValueError(f"阶梯出场缺少参数：{missing}")
    if not (0 < config.take_profit_fraction_step_0 < 1):
        raise ValueError("take_profit_fraction_step_0 必须在 0 和 1 之间。")


def apply_cash_order(
    cash: float,
    position: int,
    delta_shares: int,
    raw_price: float,
    config: StrategyConfig,
) -> tuple[float, int, float, float]:
    if delta_shares == 0:
        return cash, position, raw_price, 0.0

    slip = slippage_rate(config)
    fee = cost_rate(config)
    if delta_shares > 0:
        effective_price = raw_price * (1 + slip)
        notional = delta_shares * effective_price
        fee_paid = notional * fee
        cash -= notional + fee_paid
    else:
        effective_price = raw_price * (1 - slip)
        notional = abs(delta_shares) * effective_price
        fee_paid = notional * fee
        cash += notional - fee_paid
    return cash, position + delta_shares, effective_price, fee_paid


def base_exit_candidates(row: pd.Series, direction: int, config: StrategyConfig) -> list[ExitEvent]:
    close = float(row["close"])
    vwap_value = float(row["vwap"])
    events: list[ExitEvent] = []

    variant = config.exit_variant
    if variant in {"opposite_or_eod", "ladder"}:
        return events

    if variant in {"vwap", "vwap_and_ladder"}:
        if direction > 0 and close <= vwap_value:
            events.append(ExitEvent("vwap_exit", 0, close))
        if direction < 0 and close >= vwap_value:
            events.append(ExitEvent("vwap_exit", 0, close))
        return events

    if variant == "boundary_different_exit":
        if direction > 0 and close <= float(row["upper_exit"]):
            events.append(ExitEvent("boundary_exit", 0, close))
        if direction < 0 and close >= float(row["lower_exit"]):
            events.append(ExitEvent("boundary_exit", 0, close))
        return events

    if variant == "boundary_different_exit_and_vwap":
        if direction > 0:
            line = max(float(row["upper_exit"]), vwap_value)
            if close <= line:
                events.append(ExitEvent("boundary_vwap_exit", 0, close))
        else:
            line = min(float(row["lower_exit"]), vwap_value)
            if close >= line:
                events.append(ExitEvent("boundary_vwap_exit", 0, close))
        return events

    if variant in {"boundary_and_vwap", "boundary_and_ladder"}:
        if direction > 0:
            line = max(float(row["upper_entry"]), vwap_value)
            if close <= line:
                events.append(ExitEvent("entry_boundary_vwap_exit", 0, close))
        else:
            line = min(float(row["lower_entry"]), vwap_value)
            if close >= line:
                events.append(ExitEvent("entry_boundary_vwap_exit", 0, close))
        return events

    raise ValueError(f"未知 exit_variant：{variant}")


def ladder_levels(entry_price: float, direction: int, config: StrategyConfig) -> dict[str, float]:
    require_ladder_params(config)
    assert config.stop_loss_ladder_step_0_diff is not None
    assert config.stop_loss_ladder_step_1_diff is not None
    assert config.take_profit_ladder_step_0_diff is not None
    assert config.take_profit_ladder_step_1_diff is not None

    if direction > 0:
        return {
            "sl0": entry_price + config.stop_loss_ladder_step_0_diff,
            "sl1": entry_price + config.stop_loss_ladder_step_1_diff,
            "tp0": entry_price + config.take_profit_ladder_step_0_diff,
            "tp1": entry_price + config.take_profit_ladder_step_1_diff,
        }
    return {
        "sl0": entry_price - config.stop_loss_ladder_step_0_diff,
        "sl1": entry_price - config.stop_loss_ladder_step_1_diff,
        "tp0": entry_price - config.take_profit_ladder_step_0_diff,
        "tp1": entry_price - config.take_profit_ladder_step_1_diff,
    }


def ladder_exit_candidates(
    row: pd.Series,
    position: int,
    entry_price: float | None,
    ladder_step: int | None,
    config: StrategyConfig,
) -> list[ExitEvent]:
    if not is_ladder_variant(config) or position == 0 or entry_price is None or ladder_step is None:
        return []

    direction = sign(position)
    levels = ladder_levels(entry_price, direction, config)
    high = float(row["high"])
    low = float(row["low"])
    abs_pos = abs(position)

    if ladder_step == 0:
        if direction > 0:
            stop_hit = low <= levels["sl0"]
            tp_hit = high >= levels["tp0"]
        else:
            stop_hit = high >= levels["sl0"]
            tp_hit = low <= levels["tp0"]

        if stop_hit:
            return [
                ExitEvent(
                    "ladder_sl0_ambiguous" if tp_hit else "ladder_sl0",
                    0,
                    levels["sl0"],
                    ladder_step_after=None,
                    ambiguous=tp_hit,
                )
            ]
        if tp_hit:
            qty_to_close = max(1, int(math.floor(abs_pos * config.take_profit_fraction_step_0)))
            remaining = abs_pos - qty_to_close
            if remaining <= 0:
                return [ExitEvent("ladder_tp0_full", 0, levels["tp0"], ladder_step_after=None)]
            return [
                ExitEvent(
                    "ladder_tp0",
                    direction * remaining,
                    levels["tp0"],
                    ladder_step_after=1,
                )
            ]
        return []

    if ladder_step == 1:
        if direction > 0:
            stop_hit = low <= levels["sl1"]
            tp_hit = high >= levels["tp1"]
        else:
            stop_hit = high >= levels["sl1"]
            tp_hit = low <= levels["tp1"]

        if stop_hit:
            return [
                ExitEvent(
                    "ladder_sl1_ambiguous" if tp_hit else "ladder_sl1",
                    0,
                    levels["sl1"],
                    ladder_step_after=None,
                    ambiguous=tp_hit,
                )
            ]
        if tp_hit:
            return [ExitEvent("ladder_tp1", 0, levels["tp1"], ladder_step_after=None)]
        return []

    raise ValueError(f"未知 ladder_step：{ladder_step}")


def choose_conservative_event(events: list[ExitEvent], direction: int) -> ExitEvent | None:
    if not events:
        return None
    if direction > 0:
        return sorted(events, key=lambda event: (event.raw_price, abs(event.target_position)))[0]
    return sorted(events, key=lambda event: (-event.raw_price, abs(event.target_position)))[0]


def evaluate_exit_event(
    row: pd.Series,
    position: int,
    entry_price: float | None,
    ladder_step: int | None,
    config: StrategyConfig,
) -> ExitEvent | None:
    if position == 0:
        return None
    direction = sign(position)
    events = base_exit_candidates(row, direction, config)
    events.extend(ladder_exit_candidates(row, position, entry_price, ladder_step, config))
    return choose_conservative_event(events, direction)


In [ ]:
from __future__ import annotations

def fill_location(
    bars: pd.DataFrame,
    i: int,
    config: StrategyConfig,
    force_current_close: bool = False,
) -> tuple[int, float]:
    if force_current_close or config.execution_mode == "signal_bar_close":
        return i, float(bars.iloc[i]["close"])

    next_i = i + 1
    if next_i < len(bars) and bars.iloc[next_i]["date"] == bars.iloc[i]["date"]:
        return next_i, float(bars.iloc[next_i]["open"])
    return i, float(bars.iloc[i]["close"])


def build_order_tape(bars: pd.DataFrame, config: StrategyConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    require_ladder_params(config)
    data = bars.sort_index().reset_index(drop=True)
    index = pd.DatetimeIndex(data["timestamp"])

    order_size = pd.Series(0.0, index=index, name="order_size")
    order_price = pd.Series(data["close"].to_numpy(dtype=float), index=index, name="order_price")
    order_reason = pd.Series("", index=index, name="order_reason", dtype="object")
    target_position = pd.Series(0.0, index=index, name="target_position")
    internal_cash = pd.Series(np.nan, index=index, name="internal_cash")
    internal_value = pd.Series(np.nan, index=index, name="internal_value")
    internal_position = pd.Series(0.0, index=index, name="internal_position")

    cash = float(config.initial_equity)
    position = 0
    entry_price: float | None = None
    ladder_step: int | None = None
    day_equity = cash
    current_day = None
    pending: dict[int, tuple[int, float, str, int | None, bool]] = {}
    records: list[dict] = []

    def update_position_state(old_position: int, new_position: int, raw_price: float, step_after: int | None) -> None:
        nonlocal entry_price, ladder_step
        old_sign = sign(old_position)
        new_sign = sign(new_position)

        if new_position == 0:
            entry_price = None
            ladder_step = None
            return
        if old_position == 0 or old_sign != new_sign:
            entry_price = raw_price
            ladder_step = 0 if is_ladder_variant(config) else None
            return
        if abs(new_position) < abs(old_position) and step_after is not None:
            ladder_step = step_after

    def execute_pending(fill_i: int) -> None:
        nonlocal cash, position
        if fill_i not in pending:
            return
        target, raw_price, reason, step_after, ambiguous = pending.pop(fill_i)
        delta = int(target - position)
        if delta == 0:
            return
        old_position = position
        ts = index[fill_i]
        cash, position, effective_price, fee_paid = apply_cash_order(
            cash, position, delta, raw_price, config
        )
        update_position_state(old_position, position, raw_price, step_after)

        order_size.iloc[fill_i] += delta
        order_price.iloc[fill_i] = raw_price
        order_reason.iloc[fill_i] = reason
        records.append(
            {
                "timestamp": ts,
                "date": data.iloc[fill_i]["date"],
                "reason": reason,
                "delta_shares": delta,
                "target_position": position,
                "raw_price": raw_price,
                "effective_price": effective_price,
                "fee_paid": fee_paid,
                "cash_after_order": cash,
                "entry_price_after_order": entry_price,
                "ladder_step_after_order": ladder_step,
                "ambiguous_exit": ambiguous,
            }
        )

    def schedule_target(
        signal_i: int,
        target: int,
        reason: str,
        raw_price_override: float | None = None,
        force_current_close: bool = False,
        ladder_step_after: int | None = None,
        ambiguous: bool = False,
    ) -> None:
        if raw_price_override is None:
            fill_i, raw_price = fill_location(
                data, signal_i, config, force_current_close=force_current_close
            )
        else:
            fill_i = signal_i
            raw_price = float(raw_price_override)
        pending[fill_i] = (int(target), float(raw_price), reason, ladder_step_after, ambiguous)
        if fill_i == signal_i:
            execute_pending(fill_i)

    for i, row in data.iterrows():
        execute_pending(i)

        if current_day is None or row["date"] != current_day:
            if position != 0:
                schedule_target(i, 0, "overnight_safety_flatten", force_current_close=True)
            current_day = row["date"]
            day_equity = cash

        skip_entry_this_bar = False

        if position != 0:
            event = evaluate_exit_event(row, position, entry_price, ladder_step, config)
            if event is not None:
                schedule_target(
                    i,
                    event.target_position,
                    event.reason,
                    raw_price_override=event.raw_price,
                    ladder_step_after=event.ladder_step_after,
                    ambiguous=event.ambiguous,
                )
                skip_entry_this_bar = True
            elif bool(row["forced_exit_bar"]):
                schedule_target(i, 0, "forced_eod_exit", force_current_close=True)
                skip_entry_this_bar = True

        if not skip_entry_this_bar:
            if bool(row["entry_allowed"]):
                exposure = float(row["gross_exposure_fraction"])
                if np.isfinite(exposure) and exposure > 0:
                    _, fill_price = fill_location(data, i, config)
                    shares = int(math.floor(day_equity * exposure / fill_price))
                    if shares > 0:
                        long_signal = bool(row["long_signal"])
                        short_signal = bool(row["short_signal"])

                        if position == 0:
                            if long_signal:
                                schedule_target(i, shares, "long_entry")
                            elif short_signal:
                                schedule_target(i, -shares, "short_entry")
                        elif position > 0 and short_signal:
                            target = -shares if config.allow_reversal else 0
                            schedule_target(i, target, "long_to_short" if config.allow_reversal else "long_opposite_exit")
                        elif position < 0 and long_signal:
                            target = shares if config.allow_reversal else 0
                            schedule_target(i, target, "short_to_long" if config.allow_reversal else "short_opposite_exit")

        target_position.iloc[i] = position
        internal_position.iloc[i] = position
        internal_cash.iloc[i] = cash
        internal_value.iloc[i] = cash + position * float(row["close"])

    if position != 0:
        last_i = len(data) - 1
        schedule_target(last_i, 0, "final_safety_flatten", force_current_close=True)
        target_position.iloc[last_i] = position
        internal_position.iloc[last_i] = position
        internal_cash.iloc[last_i] = cash
        internal_value.iloc[last_i] = cash + position * float(data.iloc[last_i]["close"])

    orders = pd.DataFrame(
        {
            "close": pd.Series(data["close"].to_numpy(dtype=float), index=index),
            "order_size": order_size,
            "order_price": order_price,
            "order_reason": order_reason,
            "target_position_after_order": target_position.ffill().fillna(0.0),
            "internal_position": internal_position.ffill().fillna(0.0),
            "internal_cash": internal_cash.ffill().fillna(config.initial_equity),
            "internal_value": internal_value.ffill().fillna(config.initial_equity),
        }
    )
    records_frame = pd.DataFrame(records)
    return orders, records_frame


## 5. 回测指标

核心收益指标使用事件模拟器生成的日终内部权益。
这样可以明确保留波动率目标和杠杆敞口的行为。
同时也会基于同一份订单流水创建 vectorbt 组合对象，
方便你用熟悉的 vectorbt 方式检查订单和交易明细。


In [ ]:
from __future__ import annotations

def annualized_daily_stats(daily_equity: pd.Series) -> pd.Series:
    daily_returns = daily_equity.pct_change().dropna()
    if daily_returns.empty:
        return pd.Series(dtype=float)

    total_return = daily_equity.iloc[-1] / daily_equity.iloc[0] - 1.0
    years = max(len(daily_returns) / 252.0, 1 / 252.0)
    cagr = (1 + total_return) ** (1 / years) - 1 if total_return > -1 else np.nan
    vol = daily_returns.std(ddof=0) * np.sqrt(252)
    downside = daily_returns[daily_returns < 0].std(ddof=0) * np.sqrt(252)
    sharpe = daily_returns.mean() / daily_returns.std(ddof=0) * np.sqrt(252) if daily_returns.std(ddof=0) > 0 else np.nan
    sortino = daily_returns.mean() / downside * np.sqrt(252) if downside and downside > 0 else np.nan
    drawdown = daily_equity / daily_equity.cummax() - 1
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    return pd.Series(
        {
            "total_return": total_return,
            "cagr": cagr,
            "annualized_volatility": vol,
            "daily_sharpe": sharpe,
            "sortino": sortino,
            "max_drawdown": max_dd,
            "calmar": calmar,
            "daily_return_mean": daily_returns.mean(),
            "daily_return_std": daily_returns.std(ddof=0),
        }
    )


def add_trade_metrics(metrics: pd.Series, trades: pd.DataFrame) -> pd.Series:
    if trades.empty:
        metrics.loc["number_of_trades"] = 0
        metrics.loc["long_trades"] = 0
        metrics.loc["short_trades"] = 0
        return metrics

    pnl_col = "PnL" if "PnL" in trades.columns else None
    if pnl_col:
        wins = trades[pnl_col] > 0
        metrics.loc["win_rate"] = wins.mean()
        metrics.loc["avg_win"] = trades.loc[wins, pnl_col].mean()
        metrics.loc["avg_loss"] = trades.loc[~wins, pnl_col].mean()
        gross_profit = trades.loc[wins, pnl_col].sum()
        gross_loss = abs(trades.loc[~wins, pnl_col].sum())
        metrics.loc["profit_factor"] = gross_profit / gross_loss if gross_loss > 0 else np.nan

    metrics.loc["number_of_trades"] = len(trades)
    direction_col = "Direction" if "Direction" in trades.columns else None
    if direction_col:
        direction_text = trades[direction_col].astype(str).str.lower()
        metrics.loc["long_trades"] = direction_text.str.contains("long").sum()
        metrics.loc["short_trades"] = direction_text.str.contains("short").sum()
    return metrics


def run_vectorbt_backtest(bars: pd.DataFrame, config: StrategyConfig) -> dict:
    orders, order_records = build_order_tape(bars, config)

    # 给 vectorbt 较大的初始现金，使它能记录策略想要的杠杆订单流水。
    # leveraged order tape. Headline metrics below use internal_value.
    vectorbt_init_cash = config.initial_equity * max(1.0, config.max_leverage)
    pf = vbt.Portfolio.from_orders(
        close=orders["close"],
        size=orders["order_size"],
        price=orders["order_price"],
        size_type="amount",
        direction="both",
        init_cash=vectorbt_init_cash,
        fees=cost_rate(config),
        slippage=slippage_rate(config),
        lock_cash=False,
        allow_partial=False,
        raise_reject=False,
        freq=config.bar_size,
    )

    value = orders["internal_value"].rename("portfolio_value")
    dates = pd.to_datetime(bars.reset_index(drop=True)["date"].to_numpy())
    daily_equity = value.groupby(dates).last()
    daily_equity.index.name = "date"

    metrics = annualized_daily_stats(daily_equity)
    try:
        trades = pf.trades.records_readable
    except Exception:
        trades = pd.DataFrame()
    metrics = add_trade_metrics(metrics, trades)

    exposure = orders["internal_position"].ne(0).mean()
    metrics.loc["exposure_time_fraction"] = exposure
    turnover = orders["order_size"].abs() * orders["order_price"]
    metrics.loc["average_daily_turnover"] = turnover.groupby(dates).sum().mean()
    metrics.loc["ambiguous_ladder_exits"] = (
        int(order_records["ambiguous_exit"].sum()) if "ambiguous_exit" in order_records.columns else 0
    )
    metrics.loc["approximate_sessions"] = int(
        bars.groupby("date")["session_approximate"].max().sum()
    ) if "session_approximate" in bars.columns else 0

    monthly_returns = daily_equity.resample("M").last().pct_change().dropna()
    yearly_returns = daily_equity.resample("Y").last().pct_change().dropna()

    return {
        "portfolio": pf,
        "bars": bars,
        "orders": orders,
        "order_records": order_records,
        "daily_equity": daily_equity,
        "metrics": metrics,
        "trades": trades,
        "monthly_returns": monthly_returns,
        "yearly_returns": yearly_returns,
    }


## 6. 运行 Baseline 回测

这个单元会获取或读取 parquet 缓存数据，完成校验、特征计算，并运行 baseline 策略。
默认日期范围故意设得较短，方便先检查机制是否正常，再扩展到 2014-2024 的完整窗口。


In [ ]:
from __future__ import annotations

intraday_raw, daily_raw, dividends = load_or_fetch_spy_data(CONFIG)
bars = add_strategy_features(intraday_raw, daily_raw, dividends, CONFIG)

display(
    bars[
        [
            "date",
            "bar_time",
            "close",
            "vwap",
            "upper_entry",
            "lower_entry",
            "sigma_move",
            "entry_allowed",
            "long_signal",
            "short_signal",
            "session_approximate",
        ]
    ].head(10)
)
display(
    bars[
        ["date", "bar_time", "close", "vwap", "upper_entry", "lower_entry", "sigma_move"]
    ].tail(10)
)


In [ ]:
from __future__ import annotations

result = run_vectorbt_backtest(bars, CONFIG)
result["metrics"].to_frame("baseline")


In [ ]:
from __future__ import annotations

display(result["order_records"].head(30))
display(result["trades"].head(30))
display(result["daily_equity"].tail())


## 7. 对比更现实的执行成本

这里用同一套 baseline 逻辑重跑一次，但把执行方式改成下一根 bar 开盘成交，
并加入 spec 示例里的单边手续费和滑点。


In [ ]:
from __future__ import annotations

realistic_bars = add_strategy_features(intraday_raw, daily_raw, dividends, REALISTIC_CONFIG)
realistic_result = run_vectorbt_backtest(realistic_bars, REALISTIC_CONFIG)

comparison = pd.concat(
    [
        result["metrics"].rename("paper_style"),
        realistic_result["metrics"].rename("next_open_costed"),
    ],
    axis=1,
)
comparison


## 8. 滑点与成本敏感性

这是一个快速稳健性检查，不是参数优化。
如果极小的成本变化就能让结果翻转，说明这个信号在实盘层面太脆弱。


In [ ]:
from __future__ import annotations

cost_grid = [0.0, 0.5, 1.0, 2.0]
rows = []

for bps in cost_grid:
    cfg = replace(
        CONFIG,
        execution_mode="next_bar_open",
        transaction_cost_bps_one_way=bps,
        slippage_bps_one_way=bps,
    )
    cfg_bars = add_strategy_features(intraday_raw, daily_raw, dividends, cfg)
    cfg_result = run_vectorbt_backtest(cfg_bars, cfg)
    row = cfg_result["metrics"].copy()
    row.loc["one_way_cost_bps"] = bps
    rows.append(row)

sensitivity = pd.DataFrame(rows).set_index("one_way_cost_bps")
sensitivity[["total_return", "daily_sharpe", "max_drawdown", "number_of_trades"]]


## 9. Walk-Forward 优化框架

spec 建议使用 walk-forward 优化，而不是一次性全样本搜索。
下面这些辅助函数默认不会运行，因为它们可能很耗时，也很容易造成过拟合。
建议先确认 baseline 复现稳定，再使用这里的框架。

常见流程：

1. 一次性加载较长历史数据。
2. 定义一个较小的参数网格。
3. 每个切分窗口里，用验证集给候选参数打分。
4. 只把验证集选出的最佳参数放到后续测试窗口评估。


In [ ]:
from __future__ import annotations

def expand_param_grid(param_grid: dict[str, list]) -> list[dict]:
    import itertools

    keys = list(param_grid)
    values = [param_grid[key] for key in keys]
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]


def make_walk_forward_splits(
    all_dates: pd.Series | pd.Index,
    train_years: int = 2,
    validation_months: int = 6,
    test_months: int = 6,
    roll_forward_months: int = 6,
) -> pd.DataFrame:
    dates = pd.DatetimeIndex(pd.to_datetime(all_dates)).sort_values().unique()
    if dates.empty:
        raise ValueError("walk-forward 切分没有收到任何日期。")

    splits = []
    cursor = dates.min()
    max_date = dates.max()
    while True:
        train_start = cursor
        train_end = train_start + pd.DateOffset(years=train_years) - pd.Timedelta(days=1)
        validation_start = train_end + pd.Timedelta(days=1)
        validation_end = validation_start + pd.DateOffset(months=validation_months) - pd.Timedelta(days=1)
        test_start = validation_end + pd.Timedelta(days=1)
        test_end = test_start + pd.DateOffset(months=test_months) - pd.Timedelta(days=1)
        if test_start > max_date:
            break
        splits.append(
            {
                "train_start": train_start,
                "train_end": min(train_end, max_date),
                "validation_start": validation_start,
                "validation_end": min(validation_end, max_date),
                "test_start": test_start,
                "test_end": min(test_end, max_date),
            }
        )
        cursor = cursor + pd.DateOffset(months=roll_forward_months)
    return pd.DataFrame(splits)


def score_validation_metrics(metrics: pd.Series) -> float:
    sharpe = float(metrics.get("daily_sharpe", 0.0) or 0.0)
    cagr = float(metrics.get("cagr", 0.0) or 0.0)
    max_dd = abs(float(metrics.get("max_drawdown", 0.0) or 0.0))
    trades = float(metrics.get("number_of_trades", 0.0) or 0.0)
    if trades < 20:
        return -np.inf
    return 0.6 * sharpe + 0.3 * cagr - 0.1 * max_dd


def filter_data_window(
    intraday: pd.DataFrame,
    daily: pd.DataFrame,
    dividends: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    intraday_local_dates = parse_ny_timestamp(intraday["timestamp"]).dt.tz_convert(NY_TZ).dt.date
    intraday_dates = pd.to_datetime(intraday_local_dates)
    intraday_window = intraday.loc[(intraday_dates >= start) & (intraday_dates <= end)].copy()

    daily_dates = normalize_date_series(daily["date"])
    daily_window = daily.loc[(daily_dates >= start) & (daily_dates <= end)].copy()

    if dividends.empty:
        dividends_window = dividends.copy()
    else:
        dividend_dates = normalize_date_series(dividends["date"])
        dividends_window = dividends.loc[(dividend_dates >= start) & (dividend_dates <= end)].copy()
    return intraday_window, daily_window, dividends_window


def run_config_on_window(
    intraday: pd.DataFrame,
    daily: pd.DataFrame,
    dividends: pd.DataFrame,
    config: StrategyConfig,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> dict:
    intraday_window, daily_window, dividends_window = filter_data_window(
        intraday, daily, dividends, start, end
    )
    window_config = replace(
        config,
        start_date=pd.Timestamp(start).strftime("%Y-%m-%d"),
        end_date=pd.Timestamp(end).strftime("%Y-%m-%d"),
    )
    window_bars = add_strategy_features(intraday_window, daily_window, dividends_window, window_config)
    return run_vectorbt_backtest(window_bars, window_config)


def walk_forward_grid_search(
    intraday: pd.DataFrame,
    daily: pd.DataFrame,
    dividends: pd.DataFrame,
    base_config: StrategyConfig,
    param_grid: dict[str, list],
    splits: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    candidates = expand_param_grid(param_grid)
    for split_id, split in splits.iterrows():
        best = None
        for params in candidates:
            cfg = replace(base_config, **params)
            validation_result = run_config_on_window(
                intraday,
                daily,
                dividends,
                cfg,
                split["validation_start"],
                split["validation_end"],
            )
            score = score_validation_metrics(validation_result["metrics"])
            row = {
                "split_id": split_id,
                "stage": "validation",
                "score": score,
                **params,
                **validation_result["metrics"].add_prefix("metric_").to_dict(),
            }
            rows.append(row)
            if best is None or score > best["score"]:
                best = {"score": score, "params": params}

        if best is None or not np.isfinite(best["score"]):
            continue

        selected_cfg = replace(base_config, **best["params"])
        test_result = run_config_on_window(
            intraday,
            daily,
            dividends,
            selected_cfg,
            split["test_start"],
            split["test_end"],
        )
        rows.append(
            {
                "split_id": split_id,
                "stage": "test",
                "score": best["score"],
                **best["params"],
                **test_result["metrics"].add_prefix("metric_").to_dict(),
            }
        )
    return pd.DataFrame(rows)


# 仅作为示例；baseline 验证完成前请保持参数网格较小。
# wf_splits = make_walk_forward_splits(daily_raw["date"])
# param_grid = {
#     "lookback_days": [8, 14, 20],
#     "volatility_multiplier_entry": [0.8, 1.0, 1.2],
#     "exit_variant": ["vwap", "boundary_and_vwap"],
# }
# wf_results = walk_forward_grid_search(intraday_raw, daily_raw, dividends, CONFIG, param_grid, wf_splits)


## 10. 可选：阶梯出场变体

下面的阶梯参数只是用于检查状态机是否正常。
这些美元距离来自 spec 里的 QQQ 候选参数，所以不要把它们当成已经验证过的 SPY 参数。
这个单元适合用来确认阶梯出场的订单记录和状态变化。


In [ ]:
from __future__ import annotations

# 如需在同一批 SPY 数据上测试阶梯机制，取消下面几行注释。
# ladder_bars = add_strategy_features(intraday_raw, daily_raw, dividends, LADDER_EXAMPLE_CONFIG)
# ladder_result = run_vectorbt_backtest(ladder_bars, LADDER_EXAMPLE_CONFIG)
# display(ladder_result["metrics"].to_frame("vwap_and_ladder_example"))
# display(ladder_result["order_records"].query("reason.str.contains('ladder')", engine="python").head(20))


## 11. 导出结果

结果会写成 parquet，配置会写成 JSON。
这和仓库现有的数据缓存风格一致，也避免 CSV 对时间戳和数值列造成不必要的损失。


In [ ]:
from __future__ import annotations

def series_to_parquet(series: pd.Series, path: Path, value_name: str) -> None:
    frame = series.rename(value_name).reset_index()
    write_parquet_frame(frame, path)


def dataframe_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    if frame.empty and len(frame.columns) == 0:
        frame = pd.DataFrame({"empty": pd.Series(dtype="float64")})
    write_parquet_frame(frame, path)


def export_result(result: dict, config: StrategyConfig, label: str) -> dict[str, Path]:
    output_dir = Path(config.results_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    start = date_token(config.start_date)
    end = date_token(config.end_date)
    prefix = f"{config.symbol}_{config.bar_size}_{label}_{start}_{end}"

    paths = {
        "config": output_dir / f"{prefix}_config.json",
        "metrics": output_dir / f"{prefix}_metrics.parquet",
        "orders": output_dir / f"{prefix}_orders.parquet",
        "order_records": output_dir / f"{prefix}_order_records.parquet",
        "daily_equity": output_dir / f"{prefix}_daily_equity.parquet",
        "trades": output_dir / f"{prefix}_trades.parquet",
        "monthly_returns": output_dir / f"{prefix}_monthly_returns.parquet",
        "yearly_returns": output_dir / f"{prefix}_yearly_returns.parquet",
        "bars_sample": output_dir / f"{prefix}_bars_sample.parquet",
    }
    paths["config"].write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
    dataframe_to_parquet(result["metrics"].rename("value").reset_index(names="metric"), paths["metrics"])
    dataframe_to_parquet(result["orders"].reset_index(names="timestamp"), paths["orders"])
    dataframe_to_parquet(result["order_records"], paths["order_records"])
    series_to_parquet(result["daily_equity"], paths["daily_equity"], "portfolio_value")
    dataframe_to_parquet(result["trades"], paths["trades"])
    series_to_parquet(result["monthly_returns"], paths["monthly_returns"], "return")
    series_to_parquet(result["yearly_returns"], paths["yearly_returns"], "return")
    dataframe_to_parquet(result["bars"].head(10_000).reset_index(drop=True), paths["bars_sample"])
    return paths


written = export_result(result, CONFIG, "baseline")
written


## 12. 轻量验收测试

这些测试使用很小的合成数据，不需要 Polygon。
它们检查 spec 中风险较高的机制：VWAP 每日重置、同一时刻噪声无未来函数、
边界公式、阶梯第一步部分止盈，以及同一根 bar 同时触发止损/止盈时按保守顺序处理。


In [ ]:
from __future__ import annotations

def make_toy_intraday_for_features() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, StrategyConfig]:
    dates = pd.bdate_range("2024-01-02", periods=6)
    rows = []
    daily_rows = []
    for i, date in enumerate(dates):
        session_open = 100.0 + i
        close_1000 = session_open * (1 + 0.001 * (i + 1))
        for hhmm, price in [("09:30", session_open), ("10:00", close_1000)]:
            ts = pd.Timestamp(f"{date.date()} {hhmm}", tz=NY_TZ)
            rows.append(
                {
                    "timestamp": ts,
                    "symbol": "SPY",
                    "open": price,
                    "high": price * 1.001,
                    "low": price * 0.999,
                    "close": price,
                    "volume": 1000 + i,
                }
            )
        daily_rows.append(
            {
                "date": date,
                "symbol": "SPY",
                "open": session_open,
                "high": close_1000 * 1.01,
                "low": session_open * 0.99,
                "close": close_1000,
                "volume": 10_000,
            }
        )
    cfg = replace(
        CONFIG,
        bar_size="30min",
        lookback_days=2,
        daily_vol_window=2,
        start_trade_after_open_minutes=30,
        trade_frequency_minutes=30,
        exit_trades_before_close_minutes=5,
        missing_bar_policy="drop_day",
    )
    return pd.DataFrame(rows), pd.DataFrame(daily_rows), pd.DataFrame(columns=["date", "cash_amount"]), cfg


def test_vwap_resets() -> None:
    intraday, _, _, cfg = make_toy_intraday_for_features()
    checked = validate_intraday(intraday, cfg)
    with_vwap = add_intraday_vwap(checked)
    first_vwaps = with_vwap.groupby("date")["vwap"].first()
    first_typical = with_vwap.groupby("date").apply(
        lambda day: ((day["high"].iloc[0] + day["low"].iloc[0] + day["close"].iloc[0]) / 3.0)
    )
    assert np.allclose(first_vwaps.to_numpy(), first_typical.to_numpy())


def test_same_time_noise_and_boundary() -> None:
    intraday, daily, dividends, cfg = make_toy_intraday_for_features()
    bars = add_strategy_features(intraday, daily, dividends, cfg)
    target_date = bars["date"].drop_duplicates().iloc[3]
    row = bars[(bars["date"] == target_date) & (bars["bar_time"] == "10:00:00")].iloc[0]
    prior = bars[(bars["date"] < target_date) & (bars["bar_time"] == "10:00:00")].tail(2)
    expected_sigma = prior["abs_move_from_open"].mean()
    assert math.isclose(row["sigma_move"], expected_sigma, rel_tol=1e-12)
    expected_upper = max(row["session_open"], row["prev_close_for_boundary"]) * (1 + expected_sigma)
    assert math.isclose(row["upper_entry"], expected_upper, rel_tol=1e-12)


def make_ladder_order_bars(ambiguous: bool = False) -> pd.DataFrame:
    ts0 = pd.Timestamp("2024-01-02 10:00", tz=NY_TZ)
    ts1 = pd.Timestamp("2024-01-02 10:01", tz=NY_TZ)
    second_low = 98.5 if ambiguous else 100.0
    rows = [
        {
            "timestamp": ts0,
            "date": pd.Timestamp("2024-01-02"),
            "open": 100.0,
            "high": 100.2,
            "low": 99.8,
            "close": 100.0,
            "vwap": 99.0,
            "upper_entry": 99.5,
            "lower_entry": 98.0,
            "upper_exit": 99.5,
            "lower_exit": 98.0,
            "entry_allowed": True,
            "gross_exposure_fraction": 1.0,
            "long_signal": True,
            "short_signal": False,
            "forced_exit_bar": False,
            "session_approximate": False,
        },
        {
            "timestamp": ts1,
            "date": pd.Timestamp("2024-01-02"),
            "open": 100.0,
            "high": 101.0,
            "low": second_low,
            "close": 100.5,
            "vwap": 99.5,
            "upper_entry": 99.5,
            "lower_entry": 98.0,
            "upper_exit": 99.5,
            "lower_exit": 98.0,
            "entry_allowed": False,
            "gross_exposure_fraction": 1.0,
            "long_signal": False,
            "short_signal": False,
            "forced_exit_bar": False,
            "session_approximate": False,
        },
    ]
    return pd.DataFrame(rows).set_index("timestamp", drop=False)


def ladder_test_config() -> StrategyConfig:
    return replace(
        CONFIG,
        exit_variant="ladder",
        initial_equity=10_000.0,
        max_leverage=1.0,
        stop_loss_ladder_step_0_diff=-1.0,
        stop_loss_ladder_step_1_diff=0.0,
        take_profit_ladder_step_0_diff=0.5,
        take_profit_ladder_step_1_diff=1.5,
    )


def test_ladder_partial_take_profit() -> None:
    orders, records = build_order_tape(make_ladder_order_bars(False), ladder_test_config())
    ladder_records = records[records["reason"] == "ladder_tp0"]
    assert len(ladder_records) == 1
    assert int(ladder_records.iloc[0]["delta_shares"]) == -50
    assert int(ladder_records.iloc[0]["target_position"]) == 50


def test_ladder_ambiguous_bar_uses_stop_first() -> None:
    _, records = build_order_tape(make_ladder_order_bars(True), ladder_test_config())
    exit_record = records.iloc[-1]
    assert exit_record["reason"] == "ladder_sl0_ambiguous"
    assert bool(exit_record["ambiguous_exit"])
    assert int(exit_record["target_position"]) == 0


def run_acceptance_tests() -> None:
    test_vwap_resets()
    test_same_time_noise_and_boundary()
    test_ladder_partial_take_profit()
    test_ladder_ambiguous_bar_uses_stop_first()
    print("所有轻量验收测试已通过。")


run_acceptance_tests()


## 13. 可选图表

这里的图表故意保持简单：先确认权益曲线，再抽一天检查 close、VWAP 和上下边界，
确认边界和日内时钟对齐。


In [ ]:
from __future__ import annotations

result["daily_equity"].plot(title="SPY Intraday Momentum Daily Equity", figsize=(12, 4));


In [ ]:
from __future__ import annotations

plot_cols = ["close", "vwap", "upper_entry", "lower_entry"]
sample_date = bars["date"].drop_duplicates().iloc[min(CONFIG.lookback_days + 1, bars["date"].nunique() - 1)]
bars.loc[bars["date"].eq(sample_date), plot_cols].plot(
    title=f"SPY bands and VWAP on {sample_date.date()}",
    figsize=(12, 5),
);
